# Scheduling and Hardware Models

Compiling tells you *what* operations to run. Scheduling tells you *how long*
they take, and that answer depends entirely on the hardware you assume. This
notebook covers the scheduling strategies, the packaged hardware profiles, and
how to override individual hardware parameters.

> **Running this notebook.** From the repository root:
>
> ```bash
> uv sync
> uv run jupyter lab demo/scheduling_and_hardware.ipynb
> ```

## 1. One compile, many schedules

Scheduling does not change the circuit, so compile once and reuse it.

In [1]:
from xdqc import Compiler, Scheduler

compiler = Compiler(
    "inputs/qft_12.qasm", "inputs/ring_4qpu.json", algo="interaction"
)
compiler.compile()

print("QPUs:        ", compiler.network.num_qpus)
print("e-bit cost:  ", compiler.cost)

QPUs:         4
e-bit cost:   90.0


## 2. The four scheduling strategies

| Name | Link arbitration |
| --- | --- |
| `fifo` | Analytic pass; does **not** model entanglement generation |
| `des_link_fifo` | Discrete-event; links served first-come, first-served |
| `des_link_shortest_duration` | Discrete-event; shortest operation first |
| `des_link_critical_path` | Discrete-event; critical-path priority |

The `des_link_*` family runs a discrete-event simulation of the network: every
entangled pair has to be *generated* before the operation depending on it can
start. `fifo` skips that, so it is fast but optimistic.

In [2]:
STRATEGIES = [
    "fifo",
    "des_link_fifo",
    "des_link_shortest_duration",
    "des_link_critical_path",
]


def makespan(algo, **hardware):
    """Schedule the compiled circuit and return its makespan.

    Args:
        algo: Scheduling algorithm name.
        **hardware: Hardware selection forwarded to ``Scheduler``.

    Returns:
        The schedule makespan.
    """
    kwargs = {"algo_kwargs": {"seed": 0}} if algo != "fifo" else {}
    scheduler = Scheduler(compiler, algo=algo, **hardware, **kwargs)
    scheduler.run()
    return scheduler.schedule.makespan


hardware = {
    "modality": "neutral_atom",
    "entanglement_profile": "demo.demo",
}

print(f"{'strategy':<30} {'makespan (us)':>14}")
print("-" * 45)
for algo in STRATEGIES:
    print(f"{algo:<30} {makespan(algo, **hardware):>14.1f}")

strategy                        makespan (us)
---------------------------------------------
fifo                                    935.0
des_link_fifo                          1476.6
des_link_shortest_duration             1476.6
des_link_critical_path                 1476.6


Two things to take from this.

First, `fifo` reports a substantially **shorter** makespan than the discrete-
event schedulers. That is not `fifo` being clever — it is `fifo` not charging
for entanglement generation. Treat it as a lower bound, not a prediction.

Second, the three `des_link_*` strategies return the **same** makespan here.
Link arbitration only changes the answer when several remote operations are
ready and competing for the same link at the same instant. On these circuits
the dependency structure serializes the remote gates anyway, so there is
nothing to arbitrate. Expect them to diverge on workloads with wide, parallel
remote-gate structure — not on a QFT.

## 3. Modality: local gate times

A *modality* selects the local one- and two-qubit gate durations. The packaged
profiles come from `settings.toml`.

In [3]:
MODALITIES = ["neutral_atom", "trapped_ion.ba", "trapped_ion.sr"]

print(f"{'modality':<20} {'makespan (us)':>14}")
print("-" * 36)
for modality in MODALITIES:
    value = makespan(
        "des_link_fifo",
        modality=modality,
        entanglement_profile="demo.demo",
    )
    print(f"{modality:<20} {value:>14.1f}")

modality              makespan (us)
------------------------------------
neutral_atom                 1476.6
trapped_ion.ba              72532.0
trapped_ion.sr              31420.0


## 4. Entanglement profile: the dominant term

An *entanglement profile* sets how fast entangled pairs are generated. This is
usually the term that decides the makespan — by orders of magnitude.

In [4]:
PROFILES = [
    "demo.demo",
    "neutral_atom.polarization",
    "ion.polarization",
    "ion.time_bin",
]

print(f"{'entanglement profile':<28} {'makespan (us)':>16}")
print("-" * 46)
baseline = None
for profile in PROFILES:
    value = makespan(
        "des_link_fifo",
        modality="neutral_atom",
        entanglement_profile=profile,
    )
    baseline = baseline or value
    print(f"{profile:<28} {value:>16.1f}   ({value / baseline:>8.1f}x)")

entanglement profile            makespan (us)
----------------------------------------------
demo.demo                              1476.6   (     1.0x)
neutral_atom.polarization              3019.6   (     2.0x)
ion.polarization                      29931.4   (    20.3x)
ion.time_bin                       20860689.4   ( 14127.5x)


Both hardware axes matter, but not equally. The packaged modalities span about
**50x** in makespan — trapped-ion two-qubit gates are hundreds of times slower
than neutral-atom ones, so that is no small effect. The packaged entanglement
profiles span roughly **14,000x**.

So entanglement generation dominates the *range* of achievable execution times,
which is why partitioners are scored on e-bit count — but local gate technology
is not negligible, and the two interact. Sweep both when you characterise a
device rather than assuming one term is free.

## 5. Overriding individual hardware parameters

The named profiles are starting points. `SchedulerHardwareProfile` selects a
modality and entanglement profile *and* overrides any individual parameter.
Derived timings always recompute from the effective values, so they cannot
contradict the parameters they are built from.

In [5]:
from xdqc import SchedulerHardwareProfile

stock = SchedulerHardwareProfile.neutral_atom(entanglement_profile="demo.demo")
faster_link = SchedulerHardwareProfile.neutral_atom(
    entanglement_profile="demo.demo",
    entanglement_rate=1.0,
)
faster_gates = SchedulerHardwareProfile.neutral_atom(
    entanglement_profile="demo.demo",
    two_qubit_gate_time=0.1,
)

for label, profile in [
    ("stock neutral atom", stock),
    ("10x entanglement rate", faster_link),
    ("8x faster 2Q gate", faster_gates),
]:
    value = makespan("des_link_fifo", profile=profile)
    print(f"{label:<24} makespan={value:>10.1f} us")

stock neutral atom       makespan=    1476.6 us
10x entanglement rate    makespan=     828.0 us
8x faster 2Q gate        makespan=    1384.2 us


A ten-fold faster link buys far more than an eight-fold faster two-qubit gate —
consistent with section 4. Note that `profile=` replaces the scalar
`modality=` / `entanglement_profile=` arguments; passing both raises an error.

Factory methods exist for each packaged modality:
`SchedulerHardwareProfile.neutral_atom()`, `.ba_trapped_ion()`, and
`.sr_trapped_ion()`.

## 6. Where the numbers come from

Every profile above is read from a packaged `settings.toml`. You can load and
inspect it directly, which is the fastest way to see what a named profile
actually resolves to.

In [6]:
from xdqc import default_settings_path, load_settings

settings = load_settings()

print("settings file:", default_settings_path().name)
print("time unit:    ", settings.global_settings.time_unit)
print()

print("modality profiles")
for selector in sorted(settings.modality_profiles):
    profile = settings.modality_profile(*selector)
    print(
        f"  {'.'.join(selector):<22} "
        f"1Q={profile.one_qubit_gate_time:<8} "
        f"2Q={profile.two_qubit_gate_time}"
    )

print()
print("entanglement profiles")
for selector in sorted(settings.entanglement_profiles):
    profile = settings.entanglement_profile(*selector)
    print(
        f"  {'.'.join(selector):<28} "
        f"rate={profile.entanglement_rate:<10} "
        f"lifetime={profile.epr_lifetime}"
    )

settings file: settings.toml
time unit:     us

modality profiles
  neutral_atom           1Q=1.0      2Q=0.8
  trapped_ion.ba         1Q=10.0     2Q=500.0
  trapped_ion.sr         1Q=13.0     2Q=200.0

entanglement profiles
  demo.demo                    rate=0.1        lifetime=1000000000.0
  ion.polarization             rate=0.0025     lifetime=1000000000.0
  ion.time_bin                 rate=3.5e-06    lifetime=1000000000.0
  neutral_atom.polarization    rate=0.032      lifetime=1000000000.0


### A caveat worth knowing

Look at the `lifetime` column: every packaged profile reports an effectively
infinite EPR lifetime. The real experimental value is far shorter, but at these
generation rates a short lifetime means any operation needing *two* pairs alive
at once — a remote swap is two state teleportations — could never assemble them,
and the discrete-event schedulers would discard and regenerate forever.

So the packaged lifetime is deliberately relaxed to keep schedules terminating.
Treat absolute makespans from these profiles as **relative** comparisons rather
than hardware predictions, and override `epr_lifetime` yourself if you are
modelling a specific device.

## 7. Inspecting a schedule

Beyond the makespan, the schedule exposes every scheduled event.

In [7]:
scheduler = Scheduler(
    compiler,
    algo="des_link_fifo",
    modality="neutral_atom",
    entanglement_profile="demo.demo",
    algo_kwargs={"seed": 0},
)
scheduler.run()
schedule = scheduler.schedule

print("makespan:", round(schedule.makespan, 1), "us")
print("events:  ", len(schedule.operations))
print()

counts = {}
for event in schedule.operations:
    name = getattr(event, "kind", None) or type(event).__name__
    counts[str(name)] = counts.get(str(name), 0) + 1

print("events by kind")
for name, count in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"  {name:<24} {count}")

makespan: 1476.6 us
events:   358

events by kind
  ScheduledOperation       268
  EntanglementGeneration   90


## What you learned

- `fifo` is a lower bound; the `des_link_*` family models entanglement
  generation and is what you should quote.
- Entanglement rate spans the widest range (~14,000x across packaged
  profiles), but local gate technology still moves the makespan ~50x — sweep
  both.
- `SchedulerHardwareProfile` overrides any individual hardware parameter
  without editing `settings.toml`.

## Next

- **`visualization.ipynb`** — render the schedule as a Gantt chart and animate
  it over the network.
- **`comparing_partitioners.ipynb`** — vary the partitioner instead of the
  hardware.